# Early-Warning Feature Engineering: Day 30
This notebook rebuilds the SEAID modeling dataset using only information available on or before Day 30 of the course.  The resulting dataset is intended for realistic early-warning modeling and intervention analysis.

In [1]:
EARLY_WARNING_DAY = 30


In [2]:
print("Early-warning cutoff day:", EARLY_WARNING_DAY)


Early-warning cutoff day: 30


In [3]:
from google.colab import drive


In [4]:
drive.mount("/content/drive", force_remount=True)


Mounted at /content/drive


In [5]:
from pathlib import Path
import pandas as pd
import numpy as np


In [6]:
PROJECT_ROOT = Path("/content/drive/MyDrive/SEAID_Framework")


In [7]:
print(PROJECT_ROOT)
print(PROJECT_ROOT.exists())


/content/drive/MyDrive/SEAID_Framework
True


In [8]:
import os


In [9]:
for item in sorted(os.listdir(PROJECT_ROOT)):
    print(item)


data
figures
models
notebooks


In [10]:
from pathlib import Path


In [11]:
PROJECT_ROOT = Path("/content/drive/MyDrive/SEAID_Framework")


In [12]:
DATA_DIR = PROJECT_ROOT / "data"
FIGURES_DIR = PROJECT_ROOT / "figures"
MODELS_DIR = PROJECT_ROOT / "models"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"


In [13]:
print("Project root:", PROJECT_ROOT.exists())
print("Data folder:", DATA_DIR.exists())
print("Figures folder:", FIGURES_DIR.exists())
print("Models folder:", MODELS_DIR.exists())
print("Notebooks folder:", NOTEBOOKS_DIR.exists())


Project root: True
Data folder: True
Figures folder: True
Models folder: True
Notebooks folder: True


In [14]:
for item in sorted(DATA_DIR.iterdir()):
    print(item.name)


processed
raw


In [15]:
from pathlib import Path


In [16]:
PROJECT_ROOT = Path("/content/drive/MyDrive/SEAID_Framework")


In [17]:
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"


In [18]:
FIGURES_DIR = PROJECT_ROOT / "figures"
MODELS_DIR = PROJECT_ROOT / "models"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"


In [19]:
print("Raw data:", RAW_DATA_DIR.exists())
print("Processed data:", PROCESSED_DATA_DIR.exists())


Raw data: True
Processed data: True


In [20]:
for item in sorted(RAW_DATA_DIR.iterdir()):
    print(item.name)


assessments.csv
courses.csv
studentAssessment.csv
studentInfo.csv
studentRegistration.csv
studentVle.csv
vle.csv


In [21]:
import pandas as pd
import numpy as np


In [22]:
courses = pd.read_csv(RAW_DATA_DIR / "courses.csv")
student_info = pd.read_csv(RAW_DATA_DIR / "studentInfo.csv")
student_registration = pd.read_csv(
    RAW_DATA_DIR / "studentRegistration.csv"
)
student_assessment = pd.read_csv(
    RAW_DATA_DIR / "studentAssessment.csv"
)
assessments = pd.read_csv(RAW_DATA_DIR / "assessments.csv")
student_vle = pd.read_csv(RAW_DATA_DIR / "studentVle.csv")
vle = pd.read_csv(RAW_DATA_DIR / "vle.csv")


In [23]:
print("All OULAD datasets loaded successfully.")


All OULAD datasets loaded successfully.


In [24]:
datasets = {
    "courses": courses,
    "student_info": student_info,
    "student_registration": student_registration,
    "student_assessment": student_assessment,
    "assessments": assessments,
    "student_vle": student_vle,
    "vle": vle
}


In [25]:
for name, dataframe in datasets.items():
    print(f"{name}: {dataframe.shape}")


courses: (22, 3)
student_info: (32593, 12)
student_registration: (32593, 5)
student_assessment: (173912, 5)
assessments: (206, 6)
student_vle: (10655280, 6)
vle: (6364, 6)


In [26]:
student_features = student_info.copy()


In [27]:
print("Shape:", student_features.shape)
student_features.head()


Shape: (32593, 12)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass


In [28]:
student_keys = [
    "code_module",
    "code_presentation",
    "id_student"
]


In [29]:
duplicate_count = student_features.duplicated(
    subset=student_keys
).sum()


In [30]:
print("Duplicate student records:", duplicate_count)


Duplicate student records: 0


In [31]:
print("Starting columns:")


Starting columns:


In [32]:
for column in student_features.columns:
    print(column)


code_module
code_presentation
id_student
gender
region
highest_education
imd_band
age_band
num_of_prev_attempts
studied_credits
disability
final_result


In [33]:
student_features["final_result"].value_counts(
    dropna=False
)


,count
final_result,
Pass,12361
Withdrawn,10156
Fail,7052
Distinction,3024


In [34]:
student_features["final_result"].value_counts(
    normalize=True,
    dropna=False
).mul(100).round(2)


,proportion
final_result,
Pass,37.93
Withdrawn,31.16
Fail,21.64
Distinction,9.28


In [35]:
successful_outcomes = [
    "Pass",
    "Distinction"
]


In [36]:
student_features["target_success"] = (
    student_features["final_result"]
    .isin(successful_outcomes)
    .astype(int)
)


In [37]:
pd.crosstab(
    student_features["final_result"],
    student_features["target_success"]
)


target_success,0,1
final_result,,
Distinction,0,3024
Fail,7052,0
Pass,0,12361
Withdrawn,10156,0


In [38]:
target_distribution = (
    student_features["target_success"]
    .value_counts()
    .sort_index()
)


In [39]:
target_percentages = (
    student_features["target_success"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)


In [40]:
print("Counts:")
print(target_distribution)


Counts:
target_success
0    17208
1    15385
Name: count, dtype: int64


In [41]:
print("\nPercentages:")
print(target_percentages)



Percentages:
target_success
0    52.8
1    47.2
Name: proportion, dtype: float64


In [42]:
base_student_count = len(student_features)


In [43]:
print("Base student records:", base_student_count)


Base student records: 32593


In [44]:
print("studentVle shape:", student_vle.shape)
student_vle.head()


studentVle shape: (10655280, 6)


,code_module,code_presentation,id_student,id_site,date,sum_click
0,AAA,2013J,28400,546652,-10,4
1,AAA,2013J,28400,546652,-10,1
2,AAA,2013J,28400,546652,-10,1
3,AAA,2013J,28400,546614,-10,11
4,AAA,2013J,28400,546714,-10,1


In [45]:
print(student_vle.columns.tolist())


['code_module', 'code_presentation', 'id_student', 'id_site', 'date', 'sum_click']


In [46]:
student_vle_day30 = student_vle[
    student_vle["date"] <= EARLY_WARNING_DAY
].copy()


In [47]:
print(student_vle_day30.shape)


(2980575, 6)


In [48]:
engagement_features = (
    student_vle_day30
    .groupby(student_keys, as_index=False)
    .agg(
        total_clicks=("sum_click", "sum"),
        average_clicks_per_record=("sum_click", "mean"),
        median_clicks_per_record=("sum_click", "median"),
        maximum_clicks_in_record=("sum_click", "max"),
        vle_records=("sum_click", "size"),
        active_days=("date", "nunique"),
        unique_vle_activities=("id_site", "nunique"),
        first_activity_day=("date", "min"),
        last_activity_day=("date", "max")
    )
)


In [49]:
print("Engagement feature shape:", engagement_features.shape)
engagement_features.head()


Engagement feature shape: (28842, 12)


,code_module,code_presentation,id_student,total_clicks,average_clicks_per_record,median_clicks_per_record,maximum_clicks_in_record,vle_records,active_days,unique_vle_activities,first_activity_day,last_activity_day
0,AAA,2013J,11391,424,6.424242,2.0,76,66,10,26,-5,30
1,AAA,2013J,28400,618,3.862500,2.5,19,160,19,34,-10,28
2,AAA,2013J,30268,281,3.697368,2.0,23,76,12,22,-10,12
3,AAA,2013J,31604,540,3.552632,2.0,22,152,24,32,-10,30
4,AAA,2013J,32885,567,3.754967,2.0,22,151,24,34,-10,26


In [50]:
engagement_features["clicks_per_active_day"] = (
    engagement_features["total_clicks"]
    / engagement_features["active_days"].replace(0, np.nan)
)


In [51]:
engagement_features["activity_span_days"] = (
    engagement_features["last_activity_day"]
    - engagement_features["first_activity_day"]
)


In [52]:
engagement_features["clicks_per_vle_activity"] = (
    engagement_features["total_clicks"]
    / engagement_features["unique_vle_activities"].replace(0, np.nan)
)


In [53]:
engagement_features = engagement_features.replace(
    [np.inf, -np.inf],
    np.nan
)


In [54]:
student_features = student_features.merge(
    engagement_features,
    on=student_keys,
    how="left",
    validate="one_to_one"
)


In [55]:
print("Before merge:", base_student_count)
print("After merge:", len(student_features))


Before merge: 32593
After merge: 32593


In [56]:
engagement_zero_columns = [
    "total_clicks",
    "average_clicks_per_record",
    "median_clicks_per_record",
    "maximum_clicks_in_record",
    "vle_records",
    "active_days",
    "unique_vle_activities",
    "first_activity_day",
    "last_activity_day",
    "clicks_per_active_day",
    "activity_span_days",
    "clicks_per_vle_activity"
]


In [57]:
student_features[engagement_zero_columns] = (
    student_features[engagement_zero_columns]
    .fillna(0)
)


In [58]:
student_features["log_total_clicks"] = np.log1p(
    student_features["total_clicks"]
)


In [59]:
engagement_summary_columns = [
    "total_clicks",
    "active_days",
    "unique_vle_activities",
    "clicks_per_active_day",
    "activity_span_days",
    "log_total_clicks"
]


In [60]:
student_features[
    engagement_summary_columns
].describe().T


,count,mean,std,min,25%,50%,75%,max
total_clicks,32593.0,324.602277,432.993371,0.0,50.000000,188.000000,435.000000,8057.000000
active_days,32593.0,13.482650,11.153477,0.0,4.000000,11.000000,20.000000,56.000000
unique_vle_activities,32593.0,26.363575,22.895998,0.0,10.000000,21.000000,38.000000,290.000000
clicks_per_active_day,32593.0,18.290923,14.584830,0.0,8.684211,16.000000,25.200000,197.000000
activity_span_days,32593.0,30.135274,16.751991,0.0,20.000000,35.000000,44.000000,55.000000
log_total_clicks,32593.0,4.630631,2.101500,0.0,3.931826,5.241747,6.077642,8.994421


In [61]:
student_features[
    engagement_summary_columns
].isna().sum()


,0
total_clicks,0
active_days,0
unique_vle_activities,0
clicks_per_active_day,0
activity_span_days,0
log_total_clicks,0


In [62]:
student_assessment_enriched = (
    student_assessment.merge(
        assessments,
        on="id_assessment",
        how="left",
        validate="many_to_one"
    )
)


In [63]:
print(student_assessment_enriched.shape)
student_assessment_enriched.head()


(173912, 10)


,id_assessment,id_student,date_submitted,is_banked,score,code_module,code_presentation,assessment_type,date,weight
0,1752,11391,18,0,78.0,AAA,2013J,TMA,19.0,10.0
1,1752,28400,22,0,70.0,AAA,2013J,TMA,19.0,10.0
2,1752,31604,17,0,72.0,AAA,2013J,TMA,19.0,10.0
3,1752,32885,26,0,69.0,AAA,2013J,TMA,19.0,10.0
4,1752,38053,19,0,79.0,AAA,2013J,TMA,19.0,10.0


In [64]:
student_assessment_enriched = (
    student_assessment_enriched[
        student_assessment_enriched["date_submitted"]
        <= EARLY_WARNING_DAY
    ]
    .copy()
)


In [65]:
print(student_assessment_enriched.shape)


(26522, 10)


In [66]:
print(student_assessment_enriched.columns.tolist())


['id_assessment', 'id_student', 'date_submitted', 'is_banked', 'score', 'code_module', 'code_presentation', 'assessment_type', 'date', 'weight']


In [67]:
assessment_features = (
    student_assessment_enriched
    .groupby(student_keys, as_index=False)
    .agg(
        assessments_completed=("score", "count"),
        average_score=("score", "mean"),
        median_score=("score", "median"),
        minimum_score=("score", "min"),
        maximum_score=("score", "max"),
        score_std=("score", "std"),
        average_weight=("weight", "mean"),
        total_weight_completed=("weight", "sum"),
        first_submission_day=("date_submitted", "min"),
        last_submission_day=("date_submitted", "max"),
        banked_assessments=("is_banked", "sum")
    )
)


In [68]:
assessment_features["assessment_span_days"] = (
    assessment_features["last_submission_day"]
    - assessment_features["first_submission_day"]
)


In [69]:
assessment_features["score_range"] = (
    assessment_features["maximum_score"]
    - assessment_features["minimum_score"]
)


In [70]:
weighted_scores = (
    student_assessment_enriched
    .assign(
        weighted_points=lambda df: df["score"] * df["weight"]
    )
    .groupby(student_keys, as_index=False)
    .agg(
        weighted_points=("weighted_points", "sum"),
        total_weight=("weight", "sum")
    )
)


In [71]:
weighted_scores["weighted_average_score"] = (
    weighted_scores["weighted_points"]
    / weighted_scores["total_weight"].replace(0, np.nan)
)


In [72]:
weighted_scores = weighted_scores[
    student_keys + ["weighted_average_score"]
]


In [73]:
weighted_scores.head()


,code_module,code_presentation,id_student,weighted_average_score
0,AAA,2013J,11391,78.0
1,AAA,2013J,28400,70.0
2,AAA,2013J,31604,72.0
3,AAA,2013J,32885,69.0
4,AAA,2013J,38053,79.0


In [74]:
assessment_features = assessment_features.merge(
    weighted_scores,
    on=student_keys,
    how="left",
    validate="one_to_one"
)


In [75]:
student_features = student_features.merge(
    assessment_features,
    on=student_keys,
    how="left",
    validate="one_to_one"
)


In [76]:
print(student_features.shape)


(32593, 40)


In [77]:
assessment_columns = [
    "assessments_completed",
    "average_score",
    "median_score",
    "minimum_score",
    "maximum_score",
    "score_std",
    "average_weight",
    "total_weight_completed",
    "first_submission_day",
    "last_submission_day",
    "banked_assessments",
    "assessment_span_days",
    "score_range",
    "weighted_average_score"
]


In [78]:
student_features[assessment_columns] = (
    student_features[assessment_columns]
    .fillna(0)
)


In [79]:
student_features["score_improvement"] = (
    student_features["maximum_score"]
    - student_features["average_score"]
)


In [80]:
student_features["assessment_intensity"] = (
    student_features["assessments_completed"]
    / (student_features["assessment_span_days"] + 1)
)


In [81]:
student_features["log_assessments_completed"] = np.log1p(
    student_features["assessments_completed"]
)


In [82]:
assessment_summary = [
    "assessments_completed",
    "average_score",
    "weighted_average_score",
    "assessment_span_days",
    "assessment_intensity",
    "score_std"
]


In [83]:
student_features[assessment_summary].describe().T


,count,mean,std,min,25%,50%,75%,max
assessments_completed,32593.0,0.813150,0.809592,0.0,0.0,1.0,1.0,12.000000
average_score,32593.0,47.744662,38.987876,0.0,0.0,64.0,82.0,100.000000
weighted_average_score,32593.0,44.304969,38.697789,0.0,0.0,60.0,80.0,100.000000
assessment_span_days,32593.0,0.681926,2.826873,0.0,0.0,0.0,0.0,38.000000
assessment_intensity,32593.0,0.652598,0.679262,0.0,0.0,1.0,1.0,12.000000
score_std,32593.0,0.951094,3.591724,0.0,0.0,0.0,0.0,56.568542


In [84]:
print("studentRegistration shape:", student_registration.shape)
print(student_registration.columns.tolist())


studentRegistration shape: (32593, 5)
['code_module', 'code_presentation', 'id_student', 'date_registration', 'date_unregistration']


In [85]:
student_registration.head()


,code_module,code_presentation,id_student,date_registration,date_unregistration
0,AAA,2013J,11391,-159.0,NaN
1,AAA,2013J,28400,-53.0,NaN
2,AAA,2013J,30268,-92.0,12.0
3,AAA,2013J,31604,-52.0,NaN
4,AAA,2013J,32885,-176.0,NaN


In [86]:
registration_features = student_registration.copy()


In [87]:
registration_features["registered_before_start"] = (
    registration_features["date_registration"] < 0
).astype(int)


In [88]:
registration_features["registered_after_start"] = (
    registration_features["date_registration"] > 0
).astype(int)


In [89]:
registration_features["days_registered_before_start"] = (
    -registration_features["date_registration"]
).clip(lower=0)


In [90]:
registration_features["registration_timing"] = pd.cut(
    registration_features["date_registration"],
    bins=[
        -np.inf,
        -60,
        -30,
        -1,
        0,
        30,
        np.inf
    ],
    labels=[
        "very_early",
        "early",
        "moderately_early",
        "course_start",
        "late",
        "very_late"
    ]
)


In [91]:
registration_features["registration_timing"].value_counts(
    dropna=False
)


,count
registration_timing,
very_early,15482
early,8625
moderately_early,8205
late,218
NaN,45
very_late,16
course_start,2


In [92]:
registration_columns = student_keys + [
    "date_registration",
    "registered_before_start",
    "registered_after_start",
    "days_registered_before_start",
    "registration_timing"
]


In [93]:
registration_features = registration_features[
    registration_columns
]


In [94]:
registration_features.duplicated(
    subset=student_keys
).sum()


np.int64(0)

In [95]:
student_features = student_features.merge(
    registration_features,
    on=student_keys,
    how="left",
    validate="one_to_one"
)


In [96]:
print("Before merge:", base_student_count)
print("After merge:", len(student_features))


Before merge: 32593
After merge: 32593


In [97]:
registration_numeric_columns = [
    "date_registration",
    "registered_before_start",
    "registered_after_start",
    "days_registered_before_start"
]


In [98]:
student_features[registration_numeric_columns] = (
    student_features[registration_numeric_columns]
    .fillna(0)
)


In [99]:
student_features["registration_timing"] = (
    student_features["registration_timing"]
    .astype("object")
    .fillna("unknown")
)


In [100]:
student_features[
    registration_numeric_columns
].describe().T


,count,mean,std,min,25%,50%,75%,max
date_registration,32593.0,-69.315467,49.293930,-322.0,-100.0,-57.0,-29.0,167.0
registered_before_start,32593.0,0.991379,0.092452,0.0,1.0,1.0,1.0,1.0
registered_after_start,32593.0,0.007179,0.084428,0.0,0.0,0.0,0.0,1.0
days_registered_before_start,32593.0,69.403952,49.129177,0.0,29.0,57.0,100.0,322.0


In [101]:
student_features[
    registration_numeric_columns
    + ["registration_timing"]
].isna().sum()


,0
date_registration,0
registered_before_start,0
registered_after_start,0
days_registered_before_start,0
registration_timing,0


In [102]:
demographic_columns = [
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "disability",
    "num_of_prev_attempts",
    "studied_credits"
]


In [103]:
student_features[demographic_columns].head()


,gender,region,highest_education,imd_band,age_band,disability,num_of_prev_attempts,studied_credits
0,M,East Anglian Region,HE Qualification,90-100%,55<=,N,0,240
1,F,Scotland,HE Qualification,20-30%,35-55,N,0,60
2,F,North Western Region,A Level or Equivalent,30-40%,35-55,Y,0,60
3,F,South East Region,A Level or Equivalent,50-60%,35-55,N,0,60
4,F,West Midlands Region,Lower Than A Level,50-60%,0-35,N,0,60


In [104]:
student_features[demographic_columns].isna().sum()


,0
gender,0
region,0
highest_education,0
imd_band,1111
age_band,0
disability,0
num_of_prev_attempts,0
studied_credits,0


In [105]:
student_features["imd_band"] = (
    student_features["imd_band"]
    .fillna("Unknown")
)


In [106]:
student_features["region"] = (
    student_features["region"]
    .fillna("Unknown")
)


In [107]:
student_features["highest_education"] = (
    student_features["highest_education"]
    .fillna("Unknown")
)


In [108]:
student_features["age_band"] = (
    student_features["age_band"]
    .fillna("Unknown")
)


In [109]:
student_features["gender"] = (
    student_features["gender"]
    .fillna("Unknown")
)


In [110]:
student_features["disability"] = (
    student_features["disability"]
    .fillna("Unknown")
)


In [111]:
student_features["gender"] = (
    student_features["gender"]
    .map({
        "M": 0,
        "F": 1
    })
)


In [112]:
student_features["disability"] = (
    student_features["disability"]
    .map({
        "N": 0,
        "Y": 1
    })
)


In [113]:
categorical_columns = [
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "registration_timing"
]


In [114]:
student_features = pd.get_dummies(
    student_features,
    columns=categorical_columns,
    drop_first=True,
    dtype=int
)


In [115]:
print(student_features.shape)


(32593, 77)


In [116]:
student_features.head()


,code_module,code_presentation,id_student,gender,num_of_prev_attempts,studied_credits,disability,final_result,target_success,total_clicks,...,imd_band_90-100%,imd_band_Unknown,age_band_35-55,age_band_55<=,registration_timing_early,registration_timing_late,registration_timing_moderately_early,registration_timing_unknown,registration_timing_very_early,registration_timing_very_late
0,AAA,2013J,11391,0,0,240,0,Pass,1,424.0,...,1,0,0,1,0,0,0,0,1,0
1,AAA,2013J,28400,1,0,60,0,Pass,1,618.0,...,0,0,1,0,1,0,0,0,0,0
2,AAA,2013J,30268,1,0,60,1,Withdrawn,0,281.0,...,0,0,1,0,0,0,0,0,1,0
3,AAA,2013J,31604,1,0,60,0,Pass,1,540.0,...,0,0,1,0,1,0,0,0,0,0
4,AAA,2013J,32885,1,0,60,0,Pass,1,567.0,...,0,0,0,0,0,0,0,0,1,0


In [117]:
missing = (
    student_features
    .isna()
    .sum()
    .sort_values(ascending=False)
)


In [118]:
missing[missing > 0]


,0


In [119]:
print("Rows:", student_features.shape[0])
print("Columns:", student_features.shape[1])


Rows: 32593
Columns: 77


In [120]:
student_features.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 77 columns):
 #   Column                                         Non-Null Count  Dtype  
---  ------                                         --------------  -----  
 0   code_module                                    32593 non-null  object 
 1   code_presentation                              32593 non-null  object 
 2   id_student                                     32593 non-null  int64  
 3   gender                                         32593 non-null  int64  
 4   num_of_prev_attempts                           32593 non-null  int64  
 5   studied_credits                                32593 non-null  int64  
 6   disability                                     32593 non-null  int64  
 7   final_result                                   32593 non-null  object 
 8   target_success                                 32593 non-null  int64  
 9   total_clicks                                   325

In [121]:
leakage_columns = [
    "date_unregistration",
    "withdrew",
    "days_until_withdrawal",
    "early_withdrawal"
]


In [122]:
remaining_leakage_columns = [
    column
    for column in leakage_columns
    if column in student_features.columns
]


In [123]:
print("Leakage columns remaining:", remaining_leakage_columns)


Leakage columns remaining: []


In [124]:
assert student_features["last_activity_day"].max() <= EARLY_WARNING_DAY
assert student_features["last_submission_day"].max() <= EARLY_WARNING_DAY
assert remaining_leakage_columns == []


In [125]:
print("Day 30 cutoff and leakage checks passed.")


Day 30 cutoff and leakage checks passed.


In [126]:
output_path = (
    PROCESSED_DATA_DIR
    / "final_modeling_dataset_day30.csv"
)


In [127]:
student_features.to_csv(
    output_path,
    index=False
)


In [128]:
print("Saved to:")
print(output_path)


Saved to:
/content/drive/MyDrive/SEAID_Framework/data/processed/final_modeling_dataset_day30.csv


In [129]:
loaded = pd.read_csv(output_path)


In [130]:
print(loaded.shape)


(32593, 77)


In [131]:
loaded.head()


,code_module,code_presentation,id_student,gender,num_of_prev_attempts,studied_credits,disability,final_result,target_success,total_clicks,...,imd_band_90-100%,imd_band_Unknown,age_band_35-55,age_band_55<=,registration_timing_early,registration_timing_late,registration_timing_moderately_early,registration_timing_unknown,registration_timing_very_early,registration_timing_very_late
0,AAA,2013J,11391,0,0,240,0,Pass,1,424.0,...,1,0,0,1,0,0,0,0,1,0
1,AAA,2013J,28400,1,0,60,0,Pass,1,618.0,...,0,0,1,0,1,0,0,0,0,0
2,AAA,2013J,30268,1,0,60,1,Withdrawn,0,281.0,...,0,0,1,0,0,0,0,0,1,0
3,AAA,2013J,31604,1,0,60,0,Pass,1,540.0,...,0,0,1,0,1,0,0,0,0,0
4,AAA,2013J,32885,1,0,60,0,Pass,1,567.0,...,0,0,0,0,0,0,0,0,1,0


In [132]:
student_features = pd.read_csv(
    DATA_DIR / "processed" / "final_modeling_dataset_day30.csv"
)


In [133]:
print(student_features.shape)


(32593, 77)


In [134]:
student_features.isna().sum().sum()


np.int64(0)

In [135]:
student_features.duplicated(
    subset=student_keys
).sum()


np.int64(0)

In [136]:
feature_dictionary = pd.DataFrame({
    "Feature": student_features.columns,
    "Data Type": student_features.dtypes.astype(str),
    "Missing Values": student_features.isna().sum().values,
    "Percent Missing": (
        student_features.isna().mean().values * 100
    ).round(2)
})


In [137]:
feature_dictionary = feature_dictionary.sort_values(
    by="Feature"
).reset_index(drop=True)


In [138]:
feature_dictionary.to_csv(
    PROCESSED_DATA_DIR / "feature_dictionary_day30.csv",
    index=False
)


In [139]:
print("Feature dictionary saved to:")
print(PROCESSED_DATA_DIR / "feature_dictionary_day30.csv")
print(f"\nTotal features: {len(feature_dictionary)}")


Feature dictionary saved to:
/content/drive/MyDrive/SEAID_Framework/data/processed/feature_dictionary_day30.csv

Total features: 77


In [140]:
feature_dictionary.head()


,Feature,Data Type,Missing Values,Percent Missing
0,active_days,float64,0,0.0
1,activity_span_days,float64,0,0.0
2,age_band_35-55,int64,0,0.0
3,age_band_55<=,int64,0,0.0
4,assessment_intensity,float64,0,0.0
